This notebook explores the representation of a tetrahedron via the Python bindings to CGAL generated by swig.

The code in this notebook is based on
`https://github.com/CGAL/cgal-swig-bindings/examples/python/AABB_polyhedron_facet_intersection_example.py`.

# 1. A Tetrahedron

A tetrahedron is a polytope spanned by four points.  It is the three dimensional generalization of a triangle.

In [1]:
from __future__ import print_function
from CGAL.CGAL_Kernel import Point_3
from CGAL.CGAL_Polyhedron_3 import Polyhedron_3

In [2]:
pt1 = Point_3(1.0, 0.0, 0.0)
pt2 = Point_3(0.0, 1.0, 0.0)
pt3 = Point_3(0.0, 0.0, 1.0)
pt0 = Point_3(0.0, 0.0, 0.0)
polyhedron = Polyhedron_3()
P = polyhedron.make_tetrahedron(pt0, pt1, pt2, pt3)

In [3]:
type(P)

CGAL.CGAL_Polyhedron_3.Polyhedron_3_Halfedge_handle

Let us start with the vertices ...

In [4]:
polyhedron.size_of_vertices()

4

In [5]:
V = polyhedron.vertices()
for vertex in V:
    print(vertex.point())

0 0 1
0 1 0
1 0 0
0 0 0


No surprises here, the vertices are the points used to define the tetrahedron.

In [6]:
polyhedron.size_of_halfedges()

12

In [7]:
E = polyhedron.edges()
for e in E:
    print('edge (', e.vertex().point(), end=' , ')
    print(e.next().vertex().point(), end=') ')
    print('has degree', e.vertex_degree())

edge ( 0 0 0 , 1 0 0) has degree 3
edge ( 1 0 0 , 0 1 0) has degree 3
edge ( 0 1 0 , 0 0 0) has degree 3
edge ( 0 0 1 , 0 0 0) has degree 3
edge ( 0 0 1 , 1 0 0) has degree 3
edge ( 0 0 1 , 0 1 0) has degree 3


In [8]:
polyhedron.size_of_facets()

4

There are 4 facets (no surprises here either) and every facet is a triangle.

In [9]:
F = polyhedron.facets()
for f in F:
    print('facet is triangle?', f.is_triangle(), end=',')
    print(' with degree', f.facet_degree())

facet is triangle? True, with degree 3
facet is triangle? True, with degree 3
facet is triangle? True, with degree 3
facet is triangle? True, with degree 3


In [10]:
cnt = 0
for facet in F:
    print('facet', cnt, ':')
    e = facet.halfedge()
    print(e.vertex().point())
    print(e.next().vertex().point())
    print(e.next().next().vertex().point())
    cnt = cnt + 1

facet 0 :
0 0 0
1 0 0
0 1 0
facet 1 :
0 0 1
0 0 0
0 1 0
facet 2 :
0 0 1
1 0 0
0 0 0
facet 3 :
0 0 1
0 1 0
1 0 0


# 2. Intersection with a Line Segment

From the CGAL documentation:
The AABB tree component offers a static data structure and algorithms to perform efficient intersection and distance queries against sets of finite 3D geometric objects.

In [11]:
from CGAL.CGAL_Kernel import Segment_3
from CGAL.CGAL_AABB_tree import AABB_tree_Polyhedron_3_Facet_handle

In [12]:
tree = AABB_tree_Polyhedron_3_Facet_handle(polyhedron.facets())

Let us construct a segment.

In [13]:
a = Point_3(-0.2, 0.2, -0.2)
b = Point_3(1.3, 0.2, 1.3)
segment_query = Segment_3(a, b)

Does the segment intersect the tetrahedron?

In [14]:
tree.do_intersect(segment_query)

True

How many intersections?

In [15]:
tree.number_of_intersected_primitives(segment_query)

3

Let compute the first encountered intersection with the segment query.

In [16]:
intersection = tree.any_intersection(segment_query)
if not intersection.empty():
    op = intersection.value() # get intersection object
    object = op[0]
    if object.is_Point_3():
        print("intersection object is a point")
    print('intersects the facet spanned by')
    print(op[1].halfedge().vertex().point())
    print(op[1].halfedge().next().vertex().point())
    print(op[1].halfedge().next().next().vertex().point())

intersection object is a point
intersects the facet spanned by
0 0 0
1 0 0
0 1 0


We can ask for all intersections.

In [17]:
intersections = []
tree.all_intersections(segment_query, intersections)
for intersection in intersections:
    f = intersection[1]
    print('intersects the facet spanned by')
    print(f.halfedge().vertex().point())
    print(f.halfedge().next().vertex().point())
    print(f.halfedge().next().next().vertex().point())
    if intersection[0].is_Point_3:
        print('in a point')

intersects the facet spanned by
0 0 0
1 0 0
0 1 0
in a point
intersects the facet spanned by
0 0 1
0 0 0
0 1 0
in a point
intersects the facet spanned by
0 0 1
0 1 0
1 0 0
in a point


# 3. Intersection with a Plane

In [18]:
from CGAL.CGAL_Kernel import Vector_3
from CGAL.CGAL_Kernel import Plane_3

A plane is constructed by a point:

In [19]:
print(b)

1.3 0.2 1.3


and a vector:

In [20]:
vec = Vector_3(0.0, 1.0, 1.0)
str(vec)

'0 1 1'

In [21]:
plane_query = Plane_3(a, vec)
str(plane_query)

'0 1 1 0'

In [22]:
tree.do_intersect(plane_query)

True

Let us compute the first encountered intersection of the tetrahedron with the plane.

In [23]:
intersection = tree.any_intersection(plane_query)
if not intersection.empty():
    op = intersection.value() # get the intersection object
    object = op[0]
    if object.is_Segment_3():
        print("intersection object is a segment")
        print(op[1].halfedge().vertex().point())
        print(op[1].halfedge().next().vertex().point())

intersection object is a segment
0 0 0
1 0 0


As an exercise, compute all intersections ...